In [2]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 56.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 5.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=fe72cc9492fbc927dd0ef116d60fb71deafb3b9097dd9f2f26c196c8bb7550a2
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.
# This notebook is for a simulation of the protocol without an attacker.


In [6]:
# Helper

def quantum_random_bit():
  # generate a random bit by measuring |+>
  qc = QuantumCircuit(1, 1)
  qc.h(0)
  qc.measure(0, 0)
  backend = BasicSimulator()
  job = backend.run(transpile(qc, backend), shots=1)
  result = job.result().get_counts()
  return int(list(result.keys())[0])

def quantum_random_bits(n):
  # generate n random bits quantumly
  return [quantum_random_bit() for _ in range(n)]

In [7]:
n = 100 # no. of qubits to send

# Alice
alice_bits   = quantum_random_bits(n)   # Alice's secret bits
alice_bases  = quantum_random_bits(n)   # 0 = Z-basis, 1 = X-basis

# Alice prepares qubits and "sends" them (we store circuits)
def alice_prepare(bit, basis):
    """Prepare a single qubit: bit in given basis"""
    qc = QuantumCircuit(1, 1)
    if bit == 1:
        qc.x(0)          # flip to |1> if bit is 1
    if basis == 1:
        qc.h(0)          # rotate to X-basis if basis is X
    return qc

circuits = [alice_prepare(alice_bits[i], alice_bases[i]) for i in range(n)]
print(f"Alice bits:  {alice_bits[:10]}...")
print(f"Alice bases: {alice_bases[:10]}...  (0=Z, 1=X)")

Alice bits:  [1, 1, 0, 1, 1, 0, 1, 0, 0, 0]...
Alice bases: [0, 1, 1, 0, 0, 1, 1, 0, 0, 0]...  (0=Z, 1=X)


In [8]:
# Bob
bob_bases = quantum_random_bits(n)    # Bob's random measurement bases

def bob_measure(qc, basis):
    """Bob measures the qubit in his chosen basis"""
    qc = qc.copy()
    if basis == 1:
        qc.h(0)          # rotate back from X-basis before measuring
    qc.measure(0, 0)
    backend = BasicSimulator()
    job = backend.run(transpile(qc, backend), shots=1)
    result = job.result().get_counts()
    return int(list(result.keys())[0])

bob_results = [bob_measure(circuits[i], bob_bases[i]) for i in range(n)]
print(f"Bob bases:   {bob_bases[:10]}...  (0=Z, 1=X)")
print(f"Bob results: {bob_results[:10]}...")


Bob bases:   [1, 0, 1, 0, 0, 0, 0, 0, 1, 1]...  (0=Z, 1=X)
Bob results: [0, 1, 0, 1, 1, 1, 0, 0, 0, 1]...


In [9]:
# sifting
# Alice and Bob publicly compare bases, keep matching ones
matching = [i for i in range(n) if alice_bases[i] == bob_bases[i]]

alice_key = [alice_bits[i]   for i in matching]
bob_key   = [bob_results[i]  for i in matching]

print(f"Matching positions: {len(matching)} out of {n}")
print(f"Alice key: {alice_key[:10]}...")
print(f"Bob key:   {bob_key[:10]}...")


Matching positions: 49 out of 100
Alice key: [0, 1, 1, 0, 1, 1, 0, 0, 1, 1]...
Bob key:   [0, 1, 1, 0, 1, 1, 0, 0, 1, 1]...


In [10]:
# error check
# Sacrifice first 10 bits to check for errors
sample_size = min(10, len(alice_key))
errors = sum(alice_key[i] != bob_key[i] for i in range(sample_size))
error_rate = errors / sample_size

print(f"Checking {sample_size} bits...")
print(f"Errors: {errors}, Error rate: {error_rate:.0%}")

THRESHOLD = 0.1   # 10% error rate threshold
if error_rate > THRESHOLD:
    print("HIGH ERROR RATE — possible attacker detected! Abort.")
else:
    print("No attacker detected.")
    # Remove the sacrificed check bits, remaining bits are the key
    final_alice_key = alice_key[sample_size:]
    final_bob_key   = bob_key[sample_size:]
    print(f"\nFinal key length: {len(final_alice_key)} bits")
    print(f"Keys match: {final_alice_key == final_bob_key}")

Checking 10 bits...
Errors: 0, Error rate: 0%
No attacker detected.

Final key length: 39 bits
Keys match: True
